## Data load

In [ ]:
import pandas as pd
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


In [ ]:
df = pd.read_csv('../data/train-test.csv')
df["date"] = pd.to_datetime(df["date"])

## Split and preprocess

In [ ]:
df = df.sort_values("date")
df["month"] = df["date"].dt.month
df["dayofyear"] = df["date"].dt.dayofyear

cutoff = "2025-09-01"
train = df[df["date"] < cutoff].copy()
holdout = df[df["date"] >= cutoff].copy()
print(len(holdout)/len(train)*100,'%')

Temporal split: hold out the last 2 months of train-test (approx 24% of rows).

In [ ]:
y_train = train["posted_rate"]
y_holdout = holdout["posted_rate"]

features_december = [
    "distance", "weight", "equipment", "pickup", "delivery", "month", "dayofyear",
]
coords = ["pickup_lat", "pickup_lon", "delivery_lat", "delivery_lon"]
market = ["market_index"]
quote = ["quote_signal"]
features_all = features_december + coords + market + quote

num_dec = ["distance", "weight", "month", "dayofyear"]
num_all = num_dec + coords + market + quote
cat = ["equipment", "pickup", "delivery"]


In [ ]:
def eval_features(cols, num_cols, cat_cols, y_fit=None, inverse=None):
    y_fit = y_train if y_fit is None else y_fit
    pre = ColumnTransformer([
        ("num", SimpleImputer(strategy="median"), num_cols),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore")),
        ]), cat_cols),
    ])
    pipe = Pipeline([("preprocess", pre), ("regressor", LinearRegression())])
    pipe.fit(train[cols], y_fit)
    pred = pipe.predict(holdout[cols])
    if inverse is not None:
        pred = inverse(pred)
    mae = mean_absolute_error(y_holdout, pred)
    mse = mean_squared_error(y_holdout, pred)

    return mae, mse


## Baselines

### Training mean baseline

In [ ]:
# Baseline training mean
y_pred_mean = np.full(len(y_holdout), y_train.mean())
mae_mean = mean_absolute_error(y_holdout, y_pred_mean)
print("mean baseline MAE:", mae_mean)


### Linear baselines

In [ ]:
# Baselines with minimum available features
print("december:", eval_features(features_december, num_dec, cat))

# Baseline of all features
print("all:", eval_features(features_all, num_all, cat))


All-features beats the december-only subset on holdout (columns missing in december still carry signal).
Compare +coords vs +market/quote to see which group adds more.


In [ ]:
# Baselines of linear ablations on the same temporal holdout
print("December baseline")
print("+coords:", eval_features(features_december + coords, num_dec + coords, cat))
print("+market:", eval_features(features_december + market, num_dec + market, cat))
print("+quote:", eval_features(features_december + quote, num_dec + quote, cat))
print("+market/quote:", eval_features(features_december + market + quote, num_dec + market + quote, cat))

- coordinates don't add much relevant information if we already have cities (in our training they even add noise)
- among the extra columns, market_index carries almost all of the linear gain vs the december feature set
- quote_signal looks irrelevant (or slightly harmful) in these linear models

Linear baseline features chosen: december + market

In [ ]:
cols = features_december + market
nums = num_dec + market

print("linear $", eval_features(cols, nums, cat))
print("linear log", eval_features(cols, nums, cat, y_fit=np.log1p(y_train), inverse=np.expm1))

Predicting in log is counterproductive for the linear baseline (MAE gets much worse).

## Model

In [ ]:
cols = features_december + market
nums = num_dec + market

pre = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), nums),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ]), cat),
])

hgb = Pipeline([
    ("preprocess", pre),
    ("regressor", HistGradientBoostingRegressor(random_state=RANDOM_STATE)),
])

# dollars
hgb.fit(train[cols], y_train)
pred = hgb.predict(holdout[cols])
print("hgb $", mean_absolute_error(y_holdout, pred), mean_squared_error(y_holdout, pred))

# log -> dollars
hgb.fit(train[cols], np.log1p(y_train))
pred = np.expm1(hgb.predict(holdout[cols]))
print("hgb log", mean_absolute_error(y_holdout, pred), mean_squared_error(y_holdout, pred))


- hgb in dollars is only a small gain vs linear + market
- hgb in log improves MAE a bit more, but MSE stays similar, modest win, not a huge jump

## Brief param search

In [ ]:
cols = features_december + market
nums = num_dec + market

pre = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), nums),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ]), cat),
])

grid = [
    {"max_depth": 3, "learning_rate": 0.05, "max_iter": 200, "min_samples_leaf": 20},
    {"max_depth": 4, "learning_rate": 0.05, "max_iter": 200, "min_samples_leaf": 20},
    {"max_depth": 6, "learning_rate": 0.1,  "max_iter": 200, "min_samples_leaf": 20},
    {"max_depth": 4, "learning_rate": 0.1,  "max_iter": 300, "min_samples_leaf": 40},
]

for params in grid:
    hgb = Pipeline([
        ("preprocess", pre),
        ("regressor", HistGradientBoostingRegressor(random_state=RANDOM_STATE, **params)),
    ])
    hgb.fit(train[cols], np.log1p(y_train))
    pred = np.expm1(hgb.predict(holdout[cols]))
    print(params,
          "MAE", mean_absolute_error(y_holdout, pred),
          "MSE", mean_squared_error(y_holdout, pred))


Best HGB MAE approx 124 vs default approx 128. Gain is small. I keep the best grid config for the HGB experiment anyway

## HGB Fit and prediction on December set

In [ ]:
# final model: all labeled train-test
best_params = {
    "max_depth": 4,
    "learning_rate": 0.1,
    "max_iter": 300,
    "min_samples_leaf": 40,
}

pipe = Pipeline([
    ("preprocess", pre),
    ("regressor", HistGradientBoostingRegressor(random_state=RANDOM_STATE, **best_params)),
])

pipe.fit(df[cols], np.log1p(df["posted_rate"]))

In [ ]:
dec = pd.read_csv("../data/december-chart-inputs.csv")
dec["date"] = pd.to_datetime(dec["date"])
dec["month"] = dec["date"].dt.month
dec["dayofyear"] = dec["date"].dt.dayofyear
dec["market_index"] = np.nan   # imputer in pipe fills this

dec["predicted_rate"] = np.expm1(pipe.predict(dec[cols]))

# keep columns score.py expects
dec_out = dec[["pickup", "delivery", "distance", "equipment", "weight", "date", "predicted_rate"]]

dec_out.head()

- December predictions are flat. Hypothesis: train ends in Oct, so Dec dates are out of range.
- with a fixed load and imputed market_index, the model does not move day to day.


## Time feature fix

Tried adding a continuous `day_index` and refitting HGB (code removed).

- December `predicted_rate` stayed flat
- HGB does not extrapolate well beyond the date range seen in training (all December days fall in the same extreme time bin)
- the bottleneck is the model family, not the feature wiring
- a linear model may handle this date stress test better than HGB
- I will not add day_index in the linear model to avoid collinearity with dayofyear

## Linear check


In [ ]:
cols = features_december + market
nums = num_dec + market

print("linear $", eval_features(cols, nums, cat))

In [ ]:
cols = features_december + market
nums = num_dec + market

pipe = Pipeline([
    ("preprocess", ColumnTransformer([
        ("num", SimpleImputer(strategy="median"), nums),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore")),
        ]), cat),
    ])),
    ("regressor", LinearRegression()),
])
pipe.fit(df[cols], df["posted_rate"])  # dollars, all train-test

dec["market_index"] = np.nan
dec["predicted_rate"] = pipe.predict(dec[cols])
print(dec["predicted_rate"].nunique())
dec[["date", "predicted_rate"]].head()

- December `predicted_rate` has many unique values (date stress ok with linear + `dayofyear`, no `day_index`)
- HGB is a bit better on the holdout when time extrapolation is not the issue
- keep this linear `pipe` for both validation and December

## Final prediction

Reuse the fitted `pipe` above 

### Validation

In [ ]:
out_val = pd.DataFrame({
    "load_id": val["load_id"],
    "predicted_rate": pipe.predict(val[cols]),
})
# out_val.to_csv("../validation_predictions.csv", index=False)
out_val.head()

### December

In [ ]:
# predicted_rate already set in Linear check
dec_out = dec[["pickup", "delivery", "distance", "equipment", "weight", "date", "predicted_rate"]]
# dec_out.to_csv("../data/december-chart-inputs-predicted.csv", index=False)
dec_out.head()